In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
import pandas as pd
import numpy as np
import re
import unicodedata

from difflib import SequenceMatcher

In [2]:
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [5]:
df = pd.read_excel(r'C:\Users\alejandro_arboledas\Downloads\BASE_ENCUESTA_M0 (1).xlsx')
df

,No,DOC_ALUM,fecha diligenciado,PROGRAMA ACADEMICO,NOMBRE,EMAIL INSTITUCIONAL,TEL_CELULAR,TEL_CASA,EMAIL_PER
0,1,1004345138,2026-01-23,DISEÑO GRÁFICO,JAVIER RUIZ QUINTERO,javier.ruizq@cun.edu.co,NaN,NaN,jmaster179@hotmail.com
1,2,1000063127,2026-01-23,INGENIERÍA DE SISTEMAS,MARY ARIZA MORA,mary.ariza@cun.edu.co,3102196759,3102196759,maryariza12.07@gmail.com
2,3,1110572089,2026-01-23,INGENIERÍA INDUSTRIAL,ANGEL MENDEZ LARA,angel.mendezl@cun.edu.co,3143876733,3143876733,admendezl@ut.edu.co
3,4,1003558148,2026-01-23,INGENIERÍA INDUSTRIAL,PAULA FRÍAS PÉREZ,paula.frias@cun.edu.co,3107870105,3107870105,friaspaula176@gmail.com
4,5,1006489592,2026-01-23,ADMINISTRACIÓN DE EMPRESAS,LEIDY UNDA PADRON,Leidy.Unda@cun.edu.co,3117150791,3117150791,leidiunda1995@gmail.com
...,...,...,...,...,...,...,...,...,...
14720,14721,80726261,2026-01-29,INGENIERÍA INDUSTRIAL,JHON MORA CASTRO,jhon.moracas@cun.edu.co,573132113434,573132113434,jhonalmo@hotmail.com
14721,14722,1010053414,2026-01-29,DISEÑO GRÁFICO,NaN,NaN,NaN,NaN,NaN
14722,14723,52758004,2026-01-29,ADMINISTRACIÓN DE EMPRESAS,MARIA MERCHAN MEJIA,maria.merchanm@cun.edu.co,3219066864,3219066864,amerchan74@gmail.com
14723,14724,1088310170,2026-01-29,ESPECIALIZACIÓN EN MARKETING DIGITAL,ANDRES OSPINA QUEBRADA,andres.ospinaq@cun.edu.co,3023591218,3016725337,pipe.726@hotmail.com


In [7]:
para_orden = df['DOC_ALUM'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"


In [30]:
with engine.connect() as connection:
    try:

            query = f"""
                    SELECT DISTINCT(DOC_ALUM), PERIODO ,UBICACION, NOMBRE_FONDO, SEDE FROM ZOHO.BASE_PERSONAS 
                    WHERE UBICACION IS NOT NULL
                    AND DOC_ALUM IN ({valores_unicos_df1_orden})
                    AND PERIODO NOT LIKE '%I%'
                    AND ESTADO_PAGO = 'PAGO'
                    ORDER BY PERIODO DESC
                    
        
            """
            df_momento_2 = pd.read_sql(query, engine)
            # Agregar el resultado a la lista si no está vacío

    finally:
        
        connection.close()
df_momento_2.shape

(62070, 5)

In [31]:
df_momento_2

,DOC_ALUM,PERIODO,UBICACION,NOMBRE_FONDO,SEDE
0,1000063690,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,BOGOTA CENTRO
1,1000120872,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,BOGOTA CENTRO
2,1000128849,25V06,COLOMBIA - CUNDINAMARCA - SOACHA,BOGOTA CENTRO,BOGOTA CENTRO
3,1000133327,25V06,COLOMBIA - CUNDINAMARCA - ANOLAIMA,BOGOTA CENTRO,BOGOTA CENTRO
4,1000149937,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,BOGOTA CENTRO
...,...,...,...,...,...
62065,12637568,017V1,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,SEDE BOGOTA CENTRO
62066,17357581,017V1,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,SEDE BOGOTA CENTRO
62067,39804038,017V1,COLOMBIA - CUNDINAMARCA - CAJICA,BOGOTA CENTRO,SEDE BOGOTA CENTRO
62068,53131839,017V1,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,SEDE BOGOTA CENTRO


In [32]:
df_momento_2 = df_momento_2.drop_duplicates('DOC_ALUM')
df_momento_2

,DOC_ALUM,PERIODO,UBICACION,NOMBRE_FONDO,SEDE
0,1000063690,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,BOGOTA CENTRO
1,1000120872,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,BOGOTA CENTRO
2,1000128849,25V06,COLOMBIA - CUNDINAMARCA - SOACHA,BOGOTA CENTRO,BOGOTA CENTRO
3,1000133327,25V06,COLOMBIA - CUNDINAMARCA - ANOLAIMA,BOGOTA CENTRO,BOGOTA CENTRO
4,1000149937,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,BOGOTA CENTRO
...,...,...,...,...,...
61785,52974003,2012A,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,None
61790,53155187,2012A,COLOMBIA - CAUCA,BOGOTA CENTRO,SEDE BOGOTA CENTRO
62037,80251332,018E3,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,BOGOTA CENTRO
62038,1018424384,018E1,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO,SEDE BOGOTA CENTRO


In [33]:
df_momento_3 = df_momento_2.copy()
df_momento_3 = df_momento_3.drop(columns={'NOMBRE_FONDO'})
df_momento_3

,DOC_ALUM,PERIODO,UBICACION,SEDE
0,1000063690,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO
1,1000120872,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO
2,1000128849,25V06,COLOMBIA - CUNDINAMARCA - SOACHA,BOGOTA CENTRO
3,1000133327,25V06,COLOMBIA - CUNDINAMARCA - ANOLAIMA,BOGOTA CENTRO
4,1000149937,25V06,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO
...,...,...,...,...
61785,52974003,2012A,COLOMBIA - BOGOTÁ D.C. - CENTRO,None
61790,53155187,2012A,COLOMBIA - CAUCA,SEDE BOGOTA CENTRO
62037,80251332,018E3,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO
62038,1018424384,018E1,COLOMBIA - BOGOTÁ D.C. - CENTRO,SEDE BOGOTA CENTRO


In [34]:
df['DOC_ALUM'] = df['DOC_ALUM'].astype(str) 
df_final = df.merge(df_momento_3,how='left',on='DOC_ALUM')
df_final.drop_duplicates(inplace=True)
df_final

,No,DOC_ALUM,fecha diligenciado,PROGRAMA ACADEMICO,NOMBRE,EMAIL INSTITUCIONAL,TEL_CELULAR,TEL_CASA,EMAIL_PER,PERIODO,UBICACION,SEDE
0,1,1004345138,2026-01-23,DISEÑO GRÁFICO,JAVIER RUIZ QUINTERO,javier.ruizq@cun.edu.co,NaN,NaN,jmaster179@hotmail.com,2025C,COLOMBIA - MAGDALENA - SANTA MARTA,SANTA MARTA
1,2,1000063127,2026-01-23,INGENIERÍA DE SISTEMAS,MARY ARIZA MORA,mary.ariza@cun.edu.co,3102196759,3102196759,maryariza12.07@gmail.com,25V05,COLOMBIA - CUNDINAMARCA - BOGOTA,BOGOTA CENTRO
2,3,1110572089,2026-01-23,INGENIERÍA INDUSTRIAL,ANGEL MENDEZ LARA,angel.mendezl@cun.edu.co,3143876733,3143876733,admendezl@ut.edu.co,24V05,Colombia - Tolima - Ibague,BOGOTA CENTRO
3,4,1003558148,2026-01-23,INGENIERÍA INDUSTRIAL,PAULA FRÍAS PÉREZ,paula.frias@cun.edu.co,3107870105,3107870105,friaspaula176@gmail.com,25V04,Colombia - Cundinamarca - Tocancipa,BOGOTA CENTRO
4,5,1006489592,2026-01-23,ADMINISTRACIÓN DE EMPRESAS,LEIDY UNDA PADRON,Leidy.Unda@cun.edu.co,3117150791,3117150791,leidiunda1995@gmail.com,25V04,COLOMBIA - CASANARE - OROCUE,BOGOTA CENTRO
...,...,...,...,...,...,...,...,...,...,...,...,...
14720,14721,80726261,2026-01-29,INGENIERÍA INDUSTRIAL,JHON MORA CASTRO,jhon.moracas@cun.edu.co,573132113434,573132113434,jhonalmo@hotmail.com,24V04,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO
14721,14722,1010053414,2026-01-29,DISEÑO GRÁFICO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14722,14723,52758004,2026-01-29,ADMINISTRACIÓN DE EMPRESAS,MARIA MERCHAN MEJIA,maria.merchanm@cun.edu.co,3219066864,3219066864,amerchan74@gmail.com,25V05,COLOMBIA - BOGOTÁ D.C. - CENTRO,BOGOTA CENTRO
14723,14724,1088310170,2026-01-29,ESPECIALIZACIÓN EN MARKETING DIGITAL,ANDRES OSPINA QUEBRADA,andres.ospinaq@cun.edu.co,3023591218,3016725337,pipe.726@hotmail.com,NaN,NaN,NaN


In [35]:
df_final.to_excel(r'C:\Users\alejandro_arboledas\Downloads\BASE_ENCUESTA_M0 (2).xlsx',index=False)